In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import itertools
from skbio.diversity import alpha_diversity 
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statannotations.Annotator import Annotator
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import seaborn as sns
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
ena_meta = pl.read_csv("results/ena_meta.tsv", separator="\t").to_pandas()
ena_meta

In [ ]:
smag_meta = pl.read_excel('results/smag_metadata.xlsx',  read_options={"header_row": 1}).to_pandas()
smag_meta

In [ ]:
smag_meta = pd.merge(smag_meta,ena_meta[['run_accession','sample_accession']],left_on='Biosample', right_on='sample_accession', how='left')
smag_meta

In [ ]:
smag_diversity = pl.read_csv('./results/SMAG_all.csv').to_pandas()
smag_diversity

In [ ]:
smag_diversity = pd.merge(smag_diversity,smag_meta[['run_accession','Ecosystem', 'Nation', 'Continent']],left_on='sample', right_on='run_accession', how='left')

In [ ]:
smag_diversity[["sample","Simpson_index","trimmed_mean_chao1","Shannon_index","study","Ecosystem"]].rename(columns={'Simpson_index':'simpson', 'trimmed_mean_chao1':'chao1','Shannon_index':'shannon', 'study':'Site', 'sample':''}).set_index('').to_csv('results/SMAG_diversity.csv')

In [ ]:
shanghai_diversity = pl.read_csv('./results/shanghai_all.csv').to_pandas()
shanghai_diversity["Ecosystem"] = "Urban"

In [ ]:
df = pd.concat([shanghai_diversity, smag_diversity])
df[["sample","Simpson_index","trimmed_mean_chao1","Shannon_index","study","Ecosystem"]]

In [ ]:
custom_palette = {
    'Urban_soil': '#7570b3',
    'SMAG': '#d95f02',
}

fig, ax = plt.subplots(figsize=(4, 5))
sns.boxplot(data=df, x="study", y="trimmed_mean_chao1", color='white', hue="study", 
             palette=custom_palette,boxprops=dict(alpha=0.5),
            showfliers=False, ax=ax)

sns.swarmplot(data=df, x="study", y="trimmed_mean_chao1", hue="study", 
             palette=custom_palette, ax=ax,size=4)

tukey = pairwise_tukeyhsd(endog=df['trimmed_mean_chao1'], groups=df['study'], alpha=0.05)
summary = tukey.summary()

significant_pairs = []
for i in range(1, len(summary.data)):
    row = summary.data[i]
    if float(row[3]) < 0.05:
        group1 = row[0]
        group2 = row[1]
        p_val = float(row[3])
        significant_pairs.append( (group1, group2, p_val) )

group_order = df['study'].unique()
x_positions = {group: idx for idx, group in enumerate(group_order)}

y_max = df['trimmed_mean_chao1'].max()
y_min = df['trimmed_mean_chao1'].min()
vertical_offset = 0.1 * (y_max - y_min)
current_level = 1  
line_props = {'color': 'black', 'linewidth': 1, 'transform': ax.transData}
for pair in significant_pairs:
    group1, group2, p_val = pair
    x1 = x_positions[group1]
    x2 = x_positions[group2]
    if p_val < 0.001:
        symbol = '***'
    elif p_val < 0.01:
        symbol = '**'
    elif p_val < 0.05:
        symbol = '*'
    else:
        continue
    y_top = y_max + (vertical_offset * current_level)
    y_line = y_top - 0.1 * vertical_offset
    ax.plot([x1, x1, x2, x2], 
            [y_line, y_top, y_top, y_line], **line_props)
    ax.text((x1+x2)/2, y_top, symbol, 
            ha='center', va='bottom', 
            fontsize=12, color='black')
    current_level += 1 

ax.set_ylim(top=y_max + vertical_offset * (current_level + 1))
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylabel('Chao1', fontsize=12)
ax.set_xlabel('Study', fontsize=12)

plt.savefig("figures/chao1.svg")

In [ ]:
custom_palette = {
    'Urban_soil': '#7570b3',
    'SMAG': '#d95f02',
}

fig, ax = plt.subplots(figsize=(4, 5))
sns.boxplot(data=df, x="study", y="Shannon_index", color='white', hue="study", 
             palette=custom_palette,boxprops=dict(alpha=0.5),
            showfliers=False, ax=ax)

sns.swarmplot(data=df, x="study", y="Shannon_index", hue="study", 
             palette=custom_palette, ax=ax,size=4)

tukey = pairwise_tukeyhsd(endog=df['Shannon_index'], groups=df['study'], alpha=0.05)
summary = tukey.summary()

significant_pairs = []
for i in range(1, len(summary.data)):
    row = summary.data[i]
    if float(row[3]) < 0.05:
        group1 = row[0]
        group2 = row[1]
        p_val = float(row[3])
        significant_pairs.append( (group1, group2, p_val) )

group_order = df['study'].unique()
x_positions = {group: idx for idx, group in enumerate(group_order)}

y_max = df['Shannon_index'].max()
y_min = df['Shannon_index'].min()
vertical_offset = 0.1 * (y_max - y_min)
current_level = 1  
line_props = {'color': 'black', 'linewidth': 1, 'transform': ax.transData}
for pair in significant_pairs:
    group1, group2, p_val = pair
    x1 = x_positions[group1]
    x2 = x_positions[group2]
    if p_val < 0.001:
        symbol = '***'
    elif p_val < 0.01:
        symbol = '**'
    elif p_val < 0.05:
        symbol = '*'
    else:
        continue
    y_top = y_max + (vertical_offset * current_level)
    y_line = y_top - 0.1 * vertical_offset
    ax.plot([x1, x1, x2, x2], 
            [y_line, y_top, y_top, y_line], **line_props)
    ax.text((x1+x2)/2, y_top, symbol, 
            ha='center', va='bottom', 
            fontsize=12, color='black')
    current_level += 1 

ax.set_ylim(top=y_max + vertical_offset * (current_level + 1))
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylabel('Shannon', fontsize=12)
ax.set_xlabel('Study', fontsize=12)

plt.savefig("figures/shannon.svg")

In [ ]:
custom_palette = {
    'Urban_soil': '#7570b3',
    'SMAG': '#d95f02',
}

fig, ax = plt.subplots(figsize=(4, 5))
sns.boxplot(data=df, x="study", y="Simpson_index", color='white', hue="study", 
             palette=custom_palette,boxprops=dict(alpha=0.5),
            showfliers=False, ax=ax)

sns.swarmplot(data=df, x="study", y="Simpson_index", hue="study", 
             palette=custom_palette, ax=ax,size=2)

tukey = pairwise_tukeyhsd(endog=df['Simpson_index'], groups=df['study'], alpha=0.05)
summary = tukey.summary()

significant_pairs = []
for i in range(1, len(summary.data)):
    row = summary.data[i]
    if float(row[3]) < 0.05:
        group1 = row[0]
        group2 = row[1]
        p_val = float(row[3])
        significant_pairs.append( (group1, group2, p_val) )

group_order = df['study'].unique()
x_positions = {group: idx for idx, group in enumerate(group_order)}

y_max = df['Simpson_index'].max()
y_min = df['Simpson_index'].min()
vertical_offset = 0.1 * (y_max - y_min)
current_level = 1  
line_props = {'color': 'black', 'linewidth': 1, 'transform': ax.transData}
for pair in significant_pairs:
    group1, group2, p_val = pair
    x1 = x_positions[group1]
    x2 = x_positions[group2]
    if p_val < 0.001:
        symbol = '***'
    elif p_val < 0.01:
        symbol = '**'
    elif p_val < 0.05:
        symbol = '*'
    else:
        continue
    y_top = y_max + (vertical_offset * current_level)
    y_line = y_top - 0.1 * vertical_offset
    ax.plot([x1, x1, x2, x2], 
            [y_line, y_top, y_top, y_line], **line_props)
    ax.text((x1+x2)/2, y_top, symbol, 
            ha='center', va='bottom', 
            fontsize=12, color='black')
    current_level += 1 

ax.set_ylim(top=y_max + vertical_offset * (current_level + 1))
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylabel('Simpson', fontsize=12)
ax.set_xlabel('Study', fontsize=12)

plt.ylim(0.95, 1.05)
plt.tight_layout()
plt.savefig("figures/simpson.svg")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))

sns.boxplot(
    data=df,
    x="study",
    y="trimmed_mean_chao1",
    hue="Ecosystem",
    palette="Dark2",
    showfliers=False,
    ax=ax
)

pairs = []
pvalues = []

# Run Tukey per study
for study in df["study"].unique():

    df_sub = df[df["study"] == study]
    ecosystems = df_sub["Ecosystem"].unique()

    if len(ecosystems) < 2:
        continue  # Skip Urban_soil (only 1 ecosystem)

    tukey = pairwise_tukeyhsd(
        endog=df_sub["trimmed_mean_chao1"],
        groups=df_sub["Ecosystem"],
        alpha=0.05
    )

    results = pd.DataFrame(
        data=tukey.summary().data[1:], 
        columns=tukey.summary().data[0]
    )

    for _, row in results.iterrows():
        group1 = row["group1"]
        group2 = row["group2"]
        p_adj = float(row["p-adj"])

        pairs.append(((study, group1), (study, group2)))
        pvalues.append(p_adj)

annotator = Annotator(
    ax,
    pairs,
    data=df,
    x="study",
    y="trimmed_mean_chao1",
    hue="Ecosystem"
)

annotator.configure(
    test=None,              # IMPORTANT
    text_format='star',
    loc='inside',
    verbose=0
)

annotator.set_pvalues(pvalues)
annotator.annotate()

ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylabel('Chao1', fontsize=12)
ax.set_xlabel('Study', fontsize=12)
ax.legend(loc="upper left")

plt.tight_layout()
plt.savefig("figures/chao1_ecosystems.svg")

In [ ]:
ig, ax = plt.subplots(figsize=(8, 10))

sns.boxplot(
    data=df,
    x="study",
    y="Shannon_index",
    hue="Ecosystem",
    palette="Dark2",
    showfliers=False,
    ax=ax
)

pairs = []
pvalues = []

# Run Tukey per study
for study in df["study"].unique():

    df_sub = df[df["study"] == study]
    ecosystems = df_sub["Ecosystem"].unique()

    if len(ecosystems) < 2:
        continue  # Skip Urban_soil (only 1 ecosystem)

    tukey = pairwise_tukeyhsd(
        endog=df_sub["Shannon_index"],
        groups=df_sub["Ecosystem"],
        alpha=0.05
    )

    results = pd.DataFrame(
        data=tukey.summary().data[1:], 
        columns=tukey.summary().data[0]
    )

    for _, row in results.iterrows():
        group1 = row["group1"]
        group2 = row["group2"]
        p_adj = float(row["p-adj"])

        pairs.append(((study, group1), (study, group2)))
        pvalues.append(p_adj)

annotator = Annotator(
    ax,
    pairs,
    data=df,
    x="study",
    y="Shannon_index",
    hue="Ecosystem"
)

annotator.configure(
    test=None,              # IMPORTANT
    text_format='star',
    loc='inside',
    verbose=0
)

annotator.set_pvalues(pvalues)
annotator.annotate()

ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylabel('Shannon', fontsize=12)
ax.set_xlabel('Study', fontsize=12)
ax.legend(loc="upper left")

plt.tight_layout()
plt.savefig("figures/shannon_ecosystems.svg")

In [ ]:
ig, ax = plt.subplots(figsize=(8, 10))

sns.boxplot(
    data=df,
    x="study",
    y="Shannon_index",
    hue="Ecosystem",
    palette="Dark2",
    showfliers=False,
    ax=ax
)

pairs = []
pvalues = []

# Run Tukey per study


tukey = pairwise_tukeyhsd(
    endog=df["Shannon_index"],
    groups=df["Ecosystem"],
    alpha=0.05
)

results = pd.DataFrame(
    data=tukey.summary().data[1:], 
    columns=tukey.summary().data[0]
)

'''
for _, row in results.iterrows():
    group1 = row["group1"]
    group2 = row["group2"]
    p_adj = float(row["p-adj"])

    pairs.append( (group1, group2) )
    pvalues.append(p_adj)

annotator = Annotator(
    ax,
    pairs,
    data=df,
    x="study",
    y="Shannon_index",
    hue="Ecosystem"
)

annotator.configure(
    test=None,              # IMPORTANT
    text_format='star',
    loc='inside',
    verbose=0
)

annotator.set_pvalues(pvalues)
annotator.annotate()

ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylabel('Shannon', fontsize=12)
ax.set_xlabel('Study', fontsize=12)
ax.legend(loc="upper left")

plt.tight_layout()
plt.savefig("figures/shannon_ecosystems.svg")
'''

In [ ]:
results

In [ ]:
ig, ax = plt.subplots(figsize=(8, 10))

sns.boxplot(
    data=df,
    x="study",
    y="Simpson_index",
    hue="Ecosystem",
    palette="Dark2",
    showfliers=False,
    ax=ax
)

pairs = []
pvalues = []

# Run Tukey per study
for study in df["study"].unique():

    df_sub = df[df["study"] == study]
    ecosystems = df_sub["Ecosystem"].unique()

    if len(ecosystems) < 2:
        continue  # Skip Urban_soil (only 1 ecosystem)

    tukey = pairwise_tukeyhsd(
        endog=df_sub["Simpson_index"],
        groups=df_sub["Ecosystem"],
        alpha=0.05
    )

    results = pd.DataFrame(
        data=tukey.summary().data[1:], 
        columns=tukey.summary().data[0]
    )

    for _, row in results.iterrows():
        group1 = row["group1"]
        group2 = row["group2"]
        p_adj = float(row["p-adj"])

        pairs.append(((study, group1), (study, group2)))
        pvalues.append(p_adj)

annotator = Annotator(
    ax,
    pairs,
    data=df,
    x="study",
    y="Simpson_index",
    hue="Ecosystem"
)

annotator.configure(
    test=None,              # IMPORTANT
    text_format='star',
    loc='inside',
    verbose=0
)

annotator.set_pvalues(pvalues)
annotator.annotate()

ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylabel('Simpson', fontsize=12)
ax.set_xlabel('Study', fontsize=12)
ax.legend(loc="upper left")

plt.tight_layout()
plt.savefig("figures/simpson_ecosystems.svg")